In [19]:
import torch
import datetime
import re
import glob
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification

device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

# Whisper (ASR)
whisper_model_id = "MAdel121/whisper-small-egyptian-arabic"
whisper_pipe = pipeline(
    "automatic-speech-recognition",
    model=whisper_model_id,
    chunk_length_s=30,
    stride_length_s=5,
    device=device,
    torch_dtype=torch_dtype
)

# MARBERT (Gheeba Detection)
hate_model_id = "IbrahimAmin/marbertv2-finetuned-egyptian-hate-speech-detection"
hate_classifier = pipeline(
    "text-classification",
    model=hate_model_id,
    tokenizer=hate_model_id,
    device=device
)


def get_start_time_from_filename(filename):
    """ex: 2026-04-25_20-30-00.ogg"""
    try:
        match = re.search(r'(\d{4}-\d{2}-\d{2}_\d{2}-\d{2}-\d{2})', filename)
        if match:
            return datetime.datetime.strptime(match.group(1), '%Y-%m-%d_%H-%M-%S')
    except:
        pass
    return datetime.datetime.now()


def run_production_pipeline(audio_path):
    # Transcription
    print(f"🎙️ Starting Transcription for: {audio_path}")
    whisper_result = whisper_pipe(
        audio_path,
        batch_size=8,
        return_timestamps=True,
        generate_kwargs={"language": "arabic"}
    )

    actual_segments = whisper_result.get("chunks", [])
    start_time_obj = get_start_time_from_filename(audio_path)

    gheeba_reports = []

    # Gheeba Analysis
    print(f"🔍 Analyzing for Gheeba (Hate Speech Only)...")

    for chunk in actual_segments:
        text = chunk['text']
        start_offset = chunk['timestamp'][0]

        analysis = hate_classifier(text)[0]

        if analysis['label'] == 'Hate' and analysis['score'] > 0.65:
            actual_time = start_time_obj + datetime.timedelta(seconds=start_offset)

            gheeba_reports.append({
                "exact_time": actual_time.strftime('%H:%M:%S'),
                "text": text,
                "confidence": f"{round(analysis['score'] * 100, 2)}%"
            })

    return gheeba_reports

audio_file = glob.glob("/content/audio_test/*.ogg")[0]

try:
    final_report = run_production_pipeline(audio_file)

    if not final_report:
        print("\nResult: No Gheeba")
    else:
        print(f"\n({len(final_report)}) Gheebas detected!")
        print("-" * 50)
        for entry in final_report:
            print(f"Time: {entry['exact_time']}")
            print(f"Text: {entry['text']}")
            print(f"Accuracy: {entry['confidence']}")
            print("-" * 50)

except Exception as e:
    print(f"Error: {e}")

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: IbrahimAmin/marbertv2-finetuned-egyptian-hate-speech-detection
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🎙️ Starting Transcription for: /content/audio_test/2026-04-25_20-30-00.ogg
🔍 Analyzing for Gheeba (Hate Speech Only)...

⚠️ تقرير الغيبة (تم رصد 1 تجاوزات):
--------------------------------------------------
🕒 الوقت: 20:30:00
💬 النص:  روض محمد دا يستطع مش كويس خالس وانا بحمبوش وشوفت يستطع برابي زي
🎯 الثقة: 99.9%
--------------------------------------------------
